# Region OCR Test

Goal: improve OCR accuracy for extracting event dates from flyer images.

This notebook tests:
- region detection
- crop tuning
- preprocessing methods
- Tesseract settings

It records which methods worked best and builds toward a reusable pipeline.

In [ ]:
## 1. Setup and Imports
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pytesseract

image_dir = Path("../data/raw_images")

## 2. Region Scan Experiments
"""
Test likely flyer regions such as top, middle, and bottom to see where date text appears.
"""

for img_path in image_dir.glob("*"):
    print(f"\nProcessing: {img_path}")

    img = Image.open(img_path)

    width, height = img.size

    regions = {
        "top": (0, 0, width, int(height * 0.3)),
        "middle": (0, int(height * 0.3), width, int(height * 0.7)),
        "bottom": (0, int(height * 0.7), width, height),
    }

    # Show full image
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    # loop through regions
    for region_name, coords in regions.items():
        crop = img.crop(coords)

        print(f"--- {region_name} ---")

        plt.figure(figsize=(6, 3))
        plt.imshow(crop)
        plt.axis("off")
        plt.show()

In [3]:
## 3. Crop Tuning Experiments
"""
Test multiple crop boxes on the same flyer to find the tightest region that isolates the date with minimal noise.
"""

# Test Text Extraction on middle cropped text
crop = img.crop(regions["middle"])

crop_gray = crop.convert("L")
crop_big = crop_gray.resize((crop_gray.width * 2, crop_gray.height * 2))
crop_bw = crop_big.point(lambda p: 255 if p > 160 else 0)

text = pytesseract.image_to_string(crop_bw, config="--psm 6")

print("----- MIDDLE CROP OCR OUTPUT -----")
print(repr(text))

----- MIDDLE CROP OCR OUTPUT -----
'mere) MMOL Se : :\n; | j ’ ! Fae Phe oy .\n\nTS NEG ©, STARTING @ 73100 —\n= Cay) Le@ NS" oa ar |\n: CF 82 die *\n\n| ooh: tiga —\n'


In [12]:
## 4. Preprocessing Experiments
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pytesseract

img = Image.open("../data/raw_images/cherry_blossom_market.jpeg")

width, height = img.size

regions = {
    "top": (0, 0, width, int(height * 0.3)),
    "middle": (0, int(height * 0.3), width, int(height * 0.7)),
    "bottom": (0, int(height * 0.7), width, height),
}

crop = img.crop(regions["middle"])

plt.figure(figsize=(8, 4))
plt.imshow(crop)
plt.axis("off")
plt.show()

crop_gray = crop.convert("L")
crop_big = crop_gray.resize((crop_gray.width * 2, crop_gray.height * 2))
crop_bw = crop_big.point(lambda p: 255 if p > 160 else 0)

text = pytesseract.image_to_string(crop_bw, config="--psm 6")

print("----- MIDDLE CROP OCR OUTPUT -----")
print(repr(text))

<Figure size 800x400 with 1 Axes>

----- MIDDLE CROP OCR OUTPUT -----
'Pat Sate 7 = ors FT ee ET eo 5 ges Cee wa: pee Minna ae 2 La ‘ :\n"es eer oS oA BK is. Vi LS ee aXe pers Sear oe aye 2 un | tc, e-\n=. Ge ee a ee eee en\ncreme §\' | eer. wed gh . Gey oe Bec\ney *< 5 oo pnw PBS Bao Kk - - 4 ‘a hae T . ,, ie ~\n\nsn eA\n\nPoP AS yy TAN ag CS Pe a ne \\ eK. . DA ge : A\nf ot . 2 pat . “nye it a Fae LK oS RP. j : Yahoo ae i at?\n" . ae - Apex a won ard oT noe i f Ha f ae Be, ~ an VA ue " - \' BST\n'


In [13]:
# Test Text Extraction on tighter middle cropped text
middle_crop = img.crop(regions["middle"])

date_crop = middle_crop.crop((50, 140, 900, 450))

plt.figure(figsize=(8, 4))
plt.imshow(date_crop)
plt.axis("off")
plt.show()

crop_gray = date_crop.convert("L")
crop_big = crop_gray.resize((date_crop.width * 2, date_crop.height * 2))
crop_bw = crop_big.point(lambda p: 255 if p > 180 else 0)

text = pytesseract.image_to_string(crop_bw, config="--psm 6")

print("----- TIGHTER MIDDLE OCR OUTPUT -----")
print(repr(text))

<Figure size 800x400 with 1 Axes>

----- TIGHTER MIDDLE OCR OUTPUT -----
'~oa at CF al 6 wy ee, a ee Se ‘s; if ara . i> y mn ot ee og fae a re 2\n; ~ ESS eA er ~ FRE a TO mg ay age er AS ; ee Xe WM. ¥\noy eee ae ee he Fee Cah) en ee ee See\nEAE ES Oe cg J PRL se Wa * Pe se Be = }\nTae es Se a > ae rs * mn 7\naw* cd 2 _ <r Bs\n. eh es\naes] .\nChe A :\na te\nOsh\nay ee? * :\n® a\n\\” hoeee 2 a\n'


In [14]:
# Test Text Extraction on tighter middle cropped text
middle_crop = img.crop(regions["middle"])
date_crop = middle_crop.crop((50, 140, 900, 450))

plt.figure(figsize=(8, 4))
plt.imshow(date_crop)
plt.axis("off")
plt.show()

crop_gray = date_crop.convert("L")
crop_big = crop_gray.resize((date_crop.width * 2, date_crop.height * 2))

plt.figure(figsize=(8, 4))
plt.imshow(crop_big, cmap="gray")
plt.axis("off")
plt.show()

crop_bw = crop_big.point(lambda p: 255 if p > 210 else 0)

plt.figure(figsize=(8, 4))
plt.imshow(crop_bw, cmap="gray")
plt.axis("off")
plt.show()

text_gray = pytesseract.image_to_string(crop_big, config="--psm 6")
text_bw = pytesseract.image_to_string(crop_bw, config="--psm 6")

print("----- GRAY OCR OUTPUT -----")
print(repr(text_gray))

print("----- BW OCR OUTPUT -----")
print(repr(text_bw))

<Figure size 800x400 with 1 Axes>

<Figure size 800x400 with 1 Axes>

<Figure size 800x400 with 1 Axes>

----- GRAY OCR OUTPUT -----
'-_ FRIDAY, MARCH 27t\n4PM - 8PM\n'
----- BW OCR OUTPUT -----
'ER seem A Loa eee. ete: ‘\n: aed aN = Conary a oy var ee\nSS Ta kay. RARE 9\nom a ey pote . ney ao . he cae oe r ‘ » oe eek te x € .\' ; ‘ ; _"- er\n7 . ot Pca cers aS =) Shed © Wad 2 Pa ae :\nre niet J aN ae a ad er fees) oe\n~ ying ee i; °°: en Feet ee\n'


In [15]:
# Optional experiment: red-channel preprocessing
import cv2
import matplotlib.pyplot as plt
import pytesseract

img_cv = cv2.imread("../data/raw_images/dyke_party_01.jpg")
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)

red_channel = img_rgb[:, :, 0]

plt.figure(figsize=(6, 8))
plt.imshow(red_channel, cmap="gray")
plt.axis("off")
plt.show()

# crop with numpy slicing: [top:bottom, left:right]
red_crop = red_channel[850:1050, 0:1200]

plt.figure(figsize=(10, 4))
plt.imshow(red_crop, cmap="gray")
plt.axis("off")
plt.show()

text_red = pytesseract.image_to_string(red_crop, config="--psm 6")

print("----- RED CHANNEL OCR OUTPUT -----")
print(repr(text_red))

<Figure size 600x800 with 1 Axes>

<Figure size 1000x400 with 1 Axes>

----- RED CHANNEL OCR OUTPUT -----
'5 Co O8K aw i a yw. Zi K<j)\nay aA WF UES.\nj ZAR) Viel ©\n'


In [8]:
#Crop Again
crop = crop_bw.crop((0, 850, 1200, 1050))

plt.figure(figsize=(10,4))
plt.imshow(crop, cmap="gray")
plt.axis("off")

(np.float64(-0.5), np.float64(1199.5), np.float64(199.5), np.float64(-0.5))

<Figure size 1000x400 with 1 Axes>

In [9]:
#All together now

from PIL import Image
import matplotlib.pyplot as plt
import pytesseract
import re

img = Image.open("../data/raw_images/dyke_party_01.jpg")

# Define multiple crop options
crop_boxes = [
    (120, 930, 980, 1080),   # baseline
    (140, 930, 960, 1080),   # trim left + right
    (120, 950, 980, 1070),   # trim top + bottom
    (140, 950, 960, 1070),   # trim all sides
]

# Loop through crops and test OCR
for i, box in enumerate(crop_boxes):
    crop = img.crop(box)

    # Preprocess
    crop_bw = crop.convert("L")
    crop_bw = crop_bw.resize((crop_bw.width * 2, crop_bw.height * 2))

    # OCR
    text = pytesseract.image_to_string(
        crop_bw,
        config="--psm 7 -c tessedit_char_whitelist=0123456789/"
    )

    # 👉 CLEANING STEP GOES HERE
    text = text.replace(" ", "").strip()

    print(f"Test {i}: {box}")
    print("CLEANED OCR:", text)

    # Regex
    match = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", text)
    if match:
        month = int(match.group(1))
        day = int(match.group(2))
        year = match.group(3)

        if 1 <= month <= 12 and 1 <= day <= 31:
            print("VALID DATE FOUND:", f"{month}/{day}/{year}")
        else:
            print("INVALID DATE:", f"{month}/{day}/{year}")
    else:
        print("No clean date found")

    print("-" * 40)

Test 0: (120, 930, 980, 1080)
CLEANED OCR: 02/17/20701
VALID DATE FOUND: 2/17/2070
----------------------------------------
Test 1: (140, 930, 960, 1080)
CLEANED OCR: 62/17/2071
INVALID DATE: 62/17/2071
----------------------------------------
Test 2: (120, 950, 980, 1070)
CLEANED OCR: 6/17/46
VALID DATE FOUND: 6/17/46
----------------------------------------
Test 3: (140, 950, 960, 1070)
CLEANED OCR: 
No clean date found
----------------------------------------


In [32]:
"""
Preprocessing experiment: compare OCR accuracy across image transformations.

Tests multiple preprocessing variants on the same cropped date region:
- grayscale
- grayscale + resize
- grayscale + threshold
- grayscale + threshold + resize

Goal: identify which preprocessing method produces the most accurate OCR output
before applying regex extraction and validation.
"""
from PIL import Image
import pytesseract
import re

img = Image.open("../data/raw_images/dyke_party_01.jpg")

best_box = (120, 930, 980, 1080)
crop = img.crop(best_box)

versions = []

# Version 1: grayscale only
v1 = crop.convert("L")
versions.append(("grayscale", v1))

# Version 2: grayscale + resize
v2 = crop.convert("L")
v2 = v2.resize((v2.width * 2, v2.height * 2))
versions.append(("grayscale_resize", v2))

# Version 3: grayscale + threshold
v3 = crop.convert("L")
v3 = v3.point(lambda p: 255 if p > 160 else 0)
versions.append(("grayscale_threshold", v3))

# Version 4: grayscale + threshold + resize
v4 = crop.convert("L")
v4 = v4.point(lambda p: 255 if p > 160 else 0)
v4 = v4.resize((v4.width * 2, v4.height * 2))
versions.append(("grayscale_threshold_resize", v4))

for name, version in versions:
    text = pytesseract.image_to_string(
        version,
        config="--psm 7 -c tessedit_char_whitelist=0123456789/"
    )

    text = text.replace(" ", "").replace("\n", "").strip()

    print("VERSION:", name)
    print("CLEANED OCR:", text)

    match = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", text)

    if match:
        month = int(match.group(1))
        day = int(match.group(2))
        year = match.group(3)

        if 1 <= month <= 12 and 1 <= day <= 31:
            print("VALID DATE FOUND:", f"{month}/{day}/{year}")
        else:
            print("INVALID DATE:", f"{month}/{day}/{year}")
    else:
        print("No clean date found")

    print("-" * 40)

VERSION: grayscale
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
VERSION: grayscale_resize
CLEANED OCR: 02/17/20701
VALID DATE FOUND: 2/17/2070
----------------------------------------
VERSION: grayscale_threshold
CLEANED OCR: 062/17/20701
INVALID DATE: 62/17/2070
----------------------------------------
VERSION: grayscale_threshold_resize
CLEANED OCR: 
No clean date found
----------------------------------------


In [33]:
# Compare Tesseract PSM modes to see if layout assumptions affect OCR accuracy for date extraction
from PIL import Image
import pytesseract
import re

img = Image.open("../data/raw_images/dyke_party_01.jpg")

best_box = (120, 930, 980, 1080)
crop = img.crop(best_box)

crop_bw = crop.convert("L")

psm_modes = [6, 7, 8, 13]

for psm in psm_modes:
    config = f"--psm {psm} -c tessedit_char_whitelist=0123456789/"
    
    text = pytesseract.image_to_string(crop_bw, config=config)
    text = text.replace(" ", "").replace("\n", "").strip()

    print("PSM:", psm)
    print("CLEANED OCR:", text)

    match = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", text)

    if match:
        month = int(match.group(1))
        day = int(match.group(2))
        year = match.group(3)

        if 1 <= month <= 12 and 1 <= day <= 31:
            print("VALID DATE FOUND:", f"{month}/{day}/{year}")
        else:
            print("INVALID DATE:", f"{month}/{day}/{year}")
    else:
        print("No clean date found")

    print("-" * 40)

PSM: 6
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
PSM: 7
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
PSM: 8
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
PSM: 13
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------


## Experiment Summary

Best crop:
(120, 930, 980, 1080) — isolates date region with minimal noise

Best preprocessing:
Grayscale only — resizing and thresholding degraded OCR accuracy

PSM:
No significant difference across modes (6, 7, 8, 13)

Common OCR errors:
Year misread (e.g., 2077 instead of 2026)

Conclusion:
Use grayscale + PSM 7 + whitelist, then correct year via post-processing